# Cat vs Dog Image Classification Using 3 Convolutional Layers
**Objective:** Build, train, evaluate, and test a Deep Learning Convolutional Neural Network (CNN) with **exactly 3 Conv2D layers** for binary image classification (Cat = 0, Dog = 1).

### Key Highlights:
- **Framework:** TensorFlow 2.x / Keras
- **Input Resolution:** 128 × 128 × 3 RGB
- **Architecture:** Exactly 3 Convolutional blocks with ReLU + MaxPooling2D, followed by Flatten, Dense(128), Dropout(0.5), and Dense(1, Sigmoid)
- **Optimization:** Adam optimizer + Binary Cross-Entropy
- **Prevention of Data Leakage:** Strict train / val / test splitting (Test set held out completely)

## Step 1: Environment & Library Imports

In [ ]:
import os
import zipfile
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models

np.random.seed(42)
tf.random.set_seed(42)

print("TensorFlow Version:", tf.__version__)
gpu_devices = tf.config.list_physical_devices('GPU')
print("GPU Devices Available:", gpu_devices if gpu_devices else "No GPU detected (running on CPU)")

os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

## Step 2: Download & Extract Cats vs Dogs Dataset
We use the verified Google ML Cats and Dogs Filtered dataset (2,000 train images, 1,000 validation images).

In [ ]:
DATASET_URL = "https://storage.googleapis.com/mledu-datasets/cats_and_dogs_filtered.zip"
zip_destination = "data/cats_and_dogs_filtered.zip"
extracted_dir = "data/cats_and_dogs_filtered"

if not os.path.exists(extracted_dir):
    print("Downloading dataset archive...")
    urllib.request.urlretrieve(DATASET_URL, zip_destination)
    print("Extracting dataset archive...")
    with zipfile.ZipFile(zip_destination, 'r') as zip_ref:
        zip_ref.extractall("data")
    print("Dataset ready at:", extracted_dir)
else:
    print("Dataset already extracted at:", extracted_dir)

## Step 3: Data Pipeline (Train, Validation, and Held-out Test Sets)
To strictly avoid **data leakage**, the validation directory is divided into:
- 500 images for Validation (hyperparameter & early stopping checks)
- 500 images for Test (completely unseen evaluation)

In [ ]:
IMG_SIZE = (128, 128)
BATCH_SIZE = 32

train_dir = os.path.join(extracted_dir, "train")
val_dir = os.path.join(extracted_dir, "validation")

train_raw = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    labels="inferred",
    label_mode="binary",
    class_names=["cats", "dogs"],
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)
class_names = train_raw.class_names
print("Class Mapping:", {name: idx for idx, name in enumerate(class_names)})

val_test_raw = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    labels="inferred",
    label_mode="binary",
    class_names=["cats", "dogs"],
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=42
)

total_batches = tf.data.experimental.cardinality(val_test_raw).numpy()
val_batches = total_batches // 2

val_raw = val_test_raw.take(val_batches)
test_raw = val_test_raw.skip(val_batches)

rescaling = layers.Rescaling(1.0 / 255.0)

data_augmentation = keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1)
])

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_raw.map(lambda x, y: (rescaling(data_augmentation(x, training=True)), y), num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
val_ds = val_raw.map(lambda x, y: (rescaling(x), y), num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)
test_ds = test_raw.map(lambda x, y: (rescaling(x), y), num_parallel_calls=AUTOTUNE).cache().prefetch(AUTOTUNE)

print(f"Batches count -> Train: {tf.data.experimental.cardinality(train_raw).numpy()}, Val: {val_batches}, Test: {total_batches - val_batches}")

## Step 4: Visualizing Sample Training Images

In [ ]:
plt.figure(figsize=(9, 9))
for images, labels in train_raw.take(1):
    for i in range(9):
        ax = plt.subplot(3, 3, i + 1)
        plt.imshow(images[i].numpy().astype("uint8"))
        lbl = int(labels[i].numpy()[0])
        plt.title(f"{class_names[lbl]} ({lbl})", fontsize=11, fontweight='bold')
        plt.axis("off")
plt.tight_layout()
plt.show()

## Step 5: CNN Architecture with Exactly Three Conv2D Layers

Architecture Blueprint:
1. **Conv2D Layer 1:** 32 filters, 3×3, ReLU, Same padding + MaxPooling2D (2×2)
2. **Conv2D Layer 2:** 64 filters, 3×3, ReLU, Same padding + MaxPooling2D (2×2)
3. **Conv2D Layer 3:** 128 filters, 3×3, ReLU, Same padding + MaxPooling2D (2×2)
4. **Dense Layer:** 128 units, ReLU
5. **Dropout:** Rate = 0.5
6. **Output Layer:** 1 unit, Sigmoid

In [ ]:
def build_3layer_cnn(input_shape=(128, 128, 3), learning_rate=0.001):
    model = models.Sequential([
        layers.Input(shape=input_shape, name="input_image"),
        
        layers.Conv2D(32, (3, 3), activation="relu", padding="same", name="conv_1"),
        layers.MaxPooling2D(pool_size=(2, 2), name="pool_1"),
        
        layers.Conv2D(64, (3, 3), activation="relu", padding="same", name="conv_2"),
        layers.MaxPooling2D(pool_size=(2, 2), name="pool_2"),
        
        layers.Conv2D(128, (3, 3), activation="relu", padding="same", name="conv_3"),
        layers.MaxPooling2D(pool_size=(2, 2), name="pool_3"),
        
        layers.Flatten(name="flatten"),
        layers.Dense(128, activation="relu", name="dense_128"),
        layers.Dropout(0.5, name="dropout_50"),
        layers.Dense(1, activation="sigmoid", name="output_sigmoid")
    ], name="Cat_Dog_CNN_3Layers")
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return model

model = build_3layer_cnn(input_shape=(128, 128, 3))
model.summary()

conv_layers_count = sum(isinstance(l, layers.Conv2D) for l in model.layers)
assert conv_layers_count == 3, f"Expected 3 Conv2D layers, found {conv_layers_count}"
print(f"\nVERIFIED: The network contains EXACTLY {conv_layers_count} Conv2D layers.")

## Step 6: Model Training with EarlyStopping and ModelCheckpoint

In [ ]:
MODEL_SAVE_PATH = "models/dog_cat_cnn_model.keras"

callbacks_list = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=MODEL_SAVE_PATH,
        monitor="val_accuracy",
        save_best_only=True,
        mode="max",
        verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        verbose=1
    )
]

EPOCHS = 15
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks_list,
    verbose=1
)

## Step 7: Plotting Training and Validation Accuracy & Loss

In [ ]:
acc = history.history["accuracy"]
val_acc = history.history["val_accuracy"]
loss = history.history["loss"]
val_loss = history.history["val_loss"]
epochs_range = range(1, len(acc) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_range, acc, label="Training Accuracy", color="#1f77b4", marker="o")
axes[0].plot(epochs_range, val_acc, label="Validation Accuracy", color="#ff7f0e", marker="s")
axes[0].set_title("Training and Validation Accuracy", fontsize=12, fontweight="bold")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].grid(True, linestyle="--", alpha=0.6)
axes[0].legend(loc="lower right")

axes[1].plot(epochs_range, loss, label="Training Loss", color="#1f77b4", marker="o")
axes[1].plot(epochs_range, val_loss, label="Validation Loss", color="#ff7f0e", marker="s")
axes[1].set_title("Training and Validation Loss", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Binary Cross-Entropy Loss")
axes[1].grid(True, linestyle="--", alpha=0.6)
axes[1].legend(loc="upper right")

plt.tight_layout()
plt.savefig("results/training_history.png", dpi=300)
plt.show()

## Step 8: Evaluation on Unseen Test Set

In [ ]:
best_model = keras.models.load_model(MODEL_SAVE_PATH)

test_loss, test_acc = best_model.evaluate(test_ds, verbose=1)
print(f"\nTest Loss     : {test_loss:.4f}")
print(f"Test Accuracy : {test_acc * 100:.2f}%")

## Step 9: Confusion Matrix & Classification Report

In [ ]:
y_true_list = []
y_pred_probs_list = []

for images, labels in test_ds:
    preds = best_model.predict(images, verbose=0)
    y_true_list.extend(labels.numpy().flatten())
    y_pred_probs_list.extend(preds.flatten())

y_true = np.array(y_true_list).astype(int)
y_pred = (np.array(y_pred_probs_list) >= 0.5).astype(int)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix (Test Set)", fontweight="bold")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.savefig("results/confusion_matrix.png", dpi=300)
plt.show()

print("\nClassification Report:")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

## Step 10: Single Image Prediction Inference

In [ ]:
def predict_image(img_path):
    img = keras.utils.load_img(img_path, target_size=(128, 128))
    img_arr = keras.utils.img_to_array(img) / 255.0
    batch = np.expand_dims(img_arr, axis=0)
    prob = float(best_model.predict(batch, verbose=0)[0][0])
    label = "Dog" if prob >= 0.5 else "Cat"
    conf = (prob if prob >= 0.5 else (1.0 - prob)) * 100.0
    
    plt.imshow(img)
    plt.title(f"Prediction: {label} ({conf:.2f}%)\nRaw Prob: {prob:.4f}", fontweight='bold')
    plt.axis("off")
    plt.show()
    return label, conf

sample_cat_path = "data/cats_and_dogs_filtered/validation/cats/cat.2000.jpg"
if os.path.exists(sample_cat_path):
    predict_image(sample_cat_path)
else:
    print("Dataset path not found, upload an image to test.")

## Step 11: How to Recover if Colab Runtime Resets

If your Google Colab session disconnects or resets:
1. **Runtime Type:** Go to `Runtime` -> `Change runtime type` -> select `T4 GPU` (free & fast).
2. **Re-run Steps 1 to 3:** These will re-download the dataset and recreate the data pipelines in under 30 seconds.
3. **Persistent Checkpoints:** If you want your trained model to survive resets, mount Google Drive by adding:
   ```python
   from google.colab import drive
   drive.mount('/content/drive')
   # Save checkpoints directly to '/content/drive/MyDrive/dog_cat_cnn_model.keras'
   ```
4. **Re-load Model:** If already trained, skip Step 6 and run `best_model = keras.models.load_model('models/dog_cat_cnn_model.keras')` to proceed directly to evaluation.